In [8]:
%run "./000-config"

{'AVG_EVENTS_PER_CUSTOMER': 10,
 'CUSTOMER_MIN_VALUE': 1000,
 'DEVICE_MIN_VALUE': 1000000000,
 'K_1': 100000,
 'MB_100': 50000000,
 'NUM_DAYS': 365,
 'PLAN_MIN_VALUE': 1,
 'SUBSCRIBER_NUM_MIN_VALUE': 1000000000,
 'UNIQUE_CUSTOMERS': 50000,
 'UNIQUE_PLANS': 10,
 'VECTOR_SEARCH_ENDPOINT_NAME': 'vector_search_telco_billing',
 'agent_name': 'ai_billing_agent',
 'catalog': 'telco_billing_catalog',
 'database': 'telco_billing_db',
 'embedding_model_endpoint_name': 'databricks-gte-large-en',
 'end_dt': '2026-12-31 11:59:59',
 'llm_endpoint': 'databricks-claude-3-7-sonnet',
 'partitions_requested': 8,
 'shuffle_partitions_requested': 8,
 'start_dt': '2026-01-01 00:00:00',
 'tools_billing': 'telco_billing_catalog.telco_billing_db.lookup_billing',
 'tools_billing_faq': 'telco_billing_catalog.telco_billing_db.billing_faq',
 'tools_customer': 'telco_billing_catalog.telco_billing_db.lookup_customer',
 'tools_items': 'telco_billing_catalog.telco_billing_db.lookup_billing_items',
 'tools_plans': 'tel

### Data generation:

For this project, we will use demo data. To generate the demo data we will use data generator. The datasets include:
- billing_plans: Telcos provide multiple billing plans to their consumers. These are representative plans.
- customers: Telco customer master data including id, name, phone number, email etc.
- billing_items: The detailed billing items including device_id, event type (data, call or text), call duration, data transferred, event timestamp etc.
- invoice: Telco billing invoice for the customer

The invoice is based on:
- Cost per MB of internet activity
- Cost per minute of call for each of the call categories
- cost per message

### Databricks Labs Data Generator

In [45]:
%pip install dbldatagen

Note: you may need to restart the kernel to use updated packages.


In [11]:
# Create a catalog and a database (schema) to store the delta tables for our data
catalog = config['catalog']
db = config['database']

print(catalog, db)

telco_billing_catalog telco_billing_db


In [12]:
# Ensure the catalog existss, create if it does not exists
_ = spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")

# Ensure the schema exists within the specified catalog, create it if it does not
_ = spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{db}")


In [13]:
# Configurations for data generation
UNIQUE_CUSTOMERS = config['UNIQUE_CUSTOMERS']
CUSTOMER_MIN_VALUE = config['CUSTOMER_MIN_VALUE']
DEVICE_MIN_VALUE = config['DEVICE_MIN_VALUE']
SUBSCRIBER_NUM_MIN_VALUE = config['SUBSCRIBER_NUM_MIN_VALUE']
UNIQUE_PLANS = config['UNIQUE_PLANS']
PLAN_MIN_VALUE = config['PLAN_MIN_VALUE']

AVG_EVENTS_PER_CUSTOMER = config['AVG_EVENTS_PER_CUSTOMER']

shuffle_partitions_requested = config['shuffle_partitions_requested']
partitions_requested = config['partitions_requested']
NUM_DAYS = config['NUM_DAYS']
MB_100 = config['MB_100']
K_1 = config['K_1']
start_dt = config['start_dt']
end_dt = config['end_dt']
vector_search_index = config['catalog']

llm_endpoint = config['llm_endpoint']
warehouse_id = config['warehouse_id']

In [16]:
# Create billing plans table
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DecimalType

# Define the schema for the JSON data
schema = StructType([
    StructField("Plan_key", IntegerType(), True),
    StructField("Plan_id", StringType(), True),
    StructField("Plan_name", StringType(), True),
    StructField("contract_in_months", IntegerType(), True),
    StructField("monthly_charges_dollars", DoubleType(), True),
    StructField("Calls_Text", StringType(), True),
    StructField("Internet_Speed_MBPS", StringType(), True),
    StructField("Data_Limit_GB", StringType(), True),
    StructField("Data_Outside_Allowance_Per_MB", DoubleType(), True),
    StructField("Roam_Data_charges_per_MB", DoubleType(), True),
    StructField("Roam_Call_charges_per_min", DoubleType(), True),
    StructField("Roam_text_charges", DoubleType(), True),
    StructField("International_call_charge_per_min", DoubleType(), True),
    StructField("International_text_charge", DoubleType(), True)
])

In [28]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.workspace import ImportFormat

ws = WorkspaceClient(profile="DEV")

user = ws.current_user.me().user_name
folder = f"/Users/{user}/telco-billing-customer-care/data"
if not folder:
    ws.workspace.mkdirs(folder)

In [ ]:
import os

In [43]:
def workspace_uri(*parts: str) -> str:
    """Build a file:/Workspace/... path, even if a part starts with '/' or 'Workspace'."""
    rel = "/".join(p.strip("/") for p in parts)
    if not rel.startswith("Workspace/"):
        rel = f"Workspace/{rel}"
    return f"file:/{rel}"


plans_path = workspace_uri(folder, "billing_plans.json")
print(plans_path)  # should look like file:/Workspace/Users/<email>/.../billing_plans.json

df_plans = spark.read.format("json").schema(schema).load(plans_path)
display(df_plans)

file:/Workspace/Users/maitykalyanbrata94@gmail.com/telco-billing-customer-care/data/billing_plans.json


,Plan_key,Plan_id,Plan_name,contract_in_months,monthly_charges_dollars,Calls_Text,Internet_Speed_MBPS,Data_Limit_GB,Data_Outside_Allowance_Per_MB,Roam_Data_charges_per_MB,Roam_Call_charges_per_min,Roam_text_charges,International_call_charge_per_min,International_text_charge
0,1,PLAN001,50GB SIM12,12,25.0,UNLIMITED,100,50,0.01,0.10,1.0,0.6,0.7,0.5
1,2,PLAN002,50GB SIM24,24,22.0,UNLIMITED,100,50,0.01,0.10,1.0,0.6,0.7,0.5
2,3,PLAN003,100GB SIM12,12,28.0,UNLIMITED,100,100,0.01,0.10,1.0,0.6,0.7,0.5
3,4,PLAN004,100GB SIM24,24,26.0,UNLIMITED,100,100,0.01,0.10,1.0,0.6,0.7,0.5
4,5,PLAN005,150GB SIM12,12,30.0,UNLIMITED,100,150,0.01,0.10,1.0,0.6,0.7,0.5
5,6,PLAN006,150GB SIM24,24,27.0,UNLIMITED,100,150,0.01,0.10,1.0,0.6,0.7,0.5
6,7,PLAN007,UNLIMITED SIM12,12,35.0,UNLIMITED,UNLIMITED,UNLIMITED,0.00,0.08,0.8,0.5,0.6,0.4
7,8,PLAN008,UNLIMITED SIM24,24,32.0,UNLIMITED,UNLIMITED,UNLIMITED,0.00,0.08,0.8,0.5,0.6,0.4
8,9,PLAN009,UNLIMITED WORLD SIM12,12,42.0,UNLIMITED,UNLIMITED,UNLIMITED,0.00,0.00,0.0,0.0,0.0,0.0
9,10,PLAN010,UNLIMITED WORLD SIM24,24,38.0,UNLIMITED,UNLIMITED,UNLIMITED,0.00,0.00,0.0,0.0,0.0,0.0


### Generate customer, billing items and invoice datasets

In [49]:
%pip install jmespath pyparsing

Note: you may need to restart the kernel to use updated packages.


In [52]:
import dbldatagen as dg
import pyspark.sql.functions as F

shuffle_partitions_requested = 8
partitions_requested = 1

spark.conf.set("spark.sql.shuffle.partitions", shuffle_partitions_requested)
#spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")
#spark.conf.set("spark.sql.execution.arrow.maxRecordsPerBatch", 20000)


spark.catalog.clearCache()  # clear cache so that if we run multiple times to check performance, we're not relying on cache
shuffle_partitions_requested = 8
partitions_requested = 8
data_rows = UNIQUE_CUSTOMERS


AnalysisException: [NOT_SUPPORTED_WITH_SERVERLESS] CLEAR CACHE is not supported on serverless compute. SQLSTATE: 0A000

JVM stacktrace:
org.apache.spark.sql.AnalysisException
	at com.databricks.serverless.ServerlessGCEdgeCheck$.throwError(ServerlessGCEdgeCheck.scala:72)
	at com.databricks.serverless.ServerlessGCEdgeCheck$.checkBlockCacheCommand(ServerlessGCEdgeCheck.scala:50)
	at org.apache.spark.sql.connect.planner.SparkConnectPlanner.transformClearCache(SparkConnectPlanner.scala:6201)
	at org.apache.spark.sql.connect.planner.SparkConnectPlanner.transformCatalog(SparkConnectPlanner.scala:456)
	at org.apache.spark.sql.connect.planner.SparkConnectPlanner.$anonfun$transformRelation$3(SparkConnectPlanner.scala:368)
	at com.databricks.util.LexicalThreadLocal$Handle.runWith(LexicalThreadLocal.scala:63)
	at org.apache.spark.sql.connect.planner.SparkConnectPlanner.$anonfun$transformRelation$2(SparkConnectPlanner.scala:355)
	at org.apache.spark.sql.connect.service.SessionHolder.$anonfun$usePla